> Bash is all you need - A nano Claude Code–like agent, built from 0 to 1

- general tool design：以 codex 为例
    - https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/spec.rs
        - schema
    - bash / code:
        - bash: https://github.com/shareAI-lab/learn-claude-code/blob/main/agents/s01_agent_loop.py
        - execute code: https://github.com/UniPat-AI/SWE-Vision
        - https://mp.weixin.qq.com/s/Ei5eJssucecET-J1Lq0zLA
- 理解 agent runtime & harness，以 codex 为例
    - subagents design
    - view image
    - misc
        - apply_patch
            - 把“改代码”变成一种受限 DSL
        - write_stdin: 和一个正在运行的程序继续对话，即发送输入、获取增量输出，比如驱动 REPL（Read-Eval-Print Loop（读取-求值-输出-循环）） / interactive CLI（python/node/mysql/gdb 等）
        - exec_command 更像“一次性调用”：启动，跑一段，获得输出
            - exec model-generate code：也是通过这个 exec_command 来实现

```
python - <<'PY'
# generated code
print("hello")
PY
```

- 关于 function calling 的补充
    - https://developers.openai.com/api/docs/guides/function-calling
    - tools definition/schema: 是对入参的约束（没有显示地说明返回值），让模型学会怎样正确“发起调用”。
    - 返回结果是应用侧运行后的产物（`function_call_output`），API 把它当成“再喂给模型的一段上下文”，而不是像静态类型语言那样做编译期检查。
- responses.create 的请求体
    - `RUST_LOG='warn,codex_api=trace,codex_client=trace,codex_core=info,codex_tui=info,codex_rmcp_client=info' codex`
        - `~/.codex/logs_1.sqlite`
    - files/test_response_create_6335.json

### update 

- 04/19 2026
    - wait_agent 会阻塞主 agent，因此要少用；
    - 主 agent 如何判断 subagents 的执行状态（所谓的可见性，而非黑盒）
        - 在 MultiAgentV2 里有 list_agents，它可以列出当前 root thread tree 下 live agents 的状态：
        - 这不阻塞到完成，只是拍一次快照。主 agent 可以用它判断某个 subagent 现在是 running、completed、errored、shutdown 等。

### codex tool/function design

- https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/spec.rs
- 内部规格层：Codex 自己维护的 ToolSpec
    - 一层定义的是 Codex 内部的工具抽象，而不是直接发给 responses.create 的 JSON。在 create_spawn_agent_tool() 中，Codex 构造了一个 ToolSpec::Function(ResponsesApiTool)，里面包含：
        - name
        - description
        - parameters
        - output_schema
    - 其中 spawn_agent_output_schema() 只是这个内部 ToolSpec 的一个字段，用来描述该工具“理论上的输出结构”；它不是单独的一层，也不是最终直接发给模型的 tool definition。
- wire tool definition：真正发给 responses.create.tools（wire 电线/网线，工具在网络请求中的传输定义）
    - 这一层才是 wire layer。Codex 会把内部的 ToolSpec 通过 create_tools_json_for_responses_api() 序列化成 JSON，然后塞进 ResponsesApiRequest.tools。
    - 对于 spawn_agent，最终发给 responses.create 的通常是：
        - type
        - name
        - description
        - strict
        - parameters
    - 这里有一个很重要的细节：虽然内部定义里有 output_schema，但它不会出现在 responses.create.tools 中，因为对应字段在序列化时被跳过了。
    - 所以：
        - create_spawn_agent_tool() 定义的是内部规格
        - create_tools_json_for_responses_api() 负责生成真正的 wire tool definition
- 执行实现层：tool handler 如何真正 spawn
    - 这一层是 runtime。当模型真的发起 spawn_agent 调用后，Codex 会进入 tools/handlers/multi_agents/spawn.rs。
    - 这一层负责：
        - 解析模型传来的参数
        - 做 depth limit 校验
        - 合并/inherit 配置
        - 调用 agent_control.spawn_agent_with_options()
        - 构造 SpawnAgentResult
    - 也就是说，这一层回答的是“这个 tool 到底怎么执行”。
- 结果回注层：执行结果如何回到后续模型上下文
    - spawn_agent 执行完后，并不是直接结束，而是会把结果写成 function_call_output，再进入后续轮次的 responses.create.input。
    - Rust 内部返回值是类似：
        - agent_id
        - nickname
    - 但真正回给模型时，会被包装成 ResponseInputItem::FunctionCallOutput。因此模型下一轮看到的，不是 Rust struct，而是 function_call_output 这一类 conversation item。

### apply_patch

> 把“改代码”变成一种受限 DSL （Domain Specific Language）

- apply_patch 本身就是一个带 grammar 的 custom tool（Patch DSL）：
    - 不让模型直接“随便写文件”，让模型输出结构化 **diff**，而不是任意文本。
    - 而是强制它用 *** Begin Patch / *** Update File / @@ 这样的结构表达改动
        - codex-rs/core/src/tools/handlers/tool_apply_patch.lark
        - https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/handlers/tool_apply_patch.lark
- 收到 patch 后，handler 不会立刻改文件，而是先重新解析和验证：
    - 解析 patch 文本
    - 检查 patch grammar 是否正确
    - 推导“到底改哪些文件”
    - 推导“需要哪些写权限”
    - 决定是否需要额外审批/沙箱放权
- 为什么 llm 能理解这个 grammar or DSL
    - `cat test_response_create_6335.json| jq '.tools[4]'`
        - 调用 apply_patch 时，你要产出一段符合这份 Lark grammar 的文本。
    - https://github.com/openai/codex/blob/main/codex-rs/core/models.json

```
- Try to use apply_patch for single file edits, 
- Do not use Python to read/write files when a simple shell command or apply_patch would suffice.
```

### subagents design

- https://github.com/openai/codex/tree/main/codex-rs/core/src/tools/handlers/multi_agents
    - spawn
    - send_input：“已有子 agent 的持续交互”
    - wait
    - close：显式关闭 / shutdown 一个 sub agent。
    - resume：sub agent 可能已经结束、退出、脱离当前活跃内存态；但 parent 仍希望继续沿着那个 thread 的上下文工作
- agent name
    - https://github.com/openai/codex/blob/main/codex-rs/core/src/agent/agent_names.txt
    - 历史科学家/学者名字

- https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/spec.rs
    - spawn_agent_output_schema
    - create_spawn_agent_tool

```rust
fn spawn_agent_output_schema() -> JsonValue {
    json!({
        "type": "object",
        "properties": {
            "agent_id": {
                "type": "string",
                "description": "Thread identifier for the spawned agent."
            },
            "nickname": {
                "type": ["string", "null"],
                "description": "User-facing nickname for the spawned agent when available."
            }
        },
        "required": ["agent_id", "nickname"],
        "additionalProperties": false
    })
}
```

- spawn_agent 成功后，runtime 会把 {agent_id, nickname} 作为 function_call_output 回填到后续上下文。
- agent name 决策逻辑
    - 先看 role 有没有配置专属 nickname_candidates（`config.toml`）
    - 如果有，就从这个候选池里选
    - 如果没有，就退回默认的内置名字池 agent_names.txt
    - 分配时尽量避开当前活跃 agent 已经占用的名字

```
llm 决策触发工具调用：
{ "type": "function_call", "name": "spawn_agent", ... }

runtime 执行后，后续上下文里出现：
{
  "type": "function_call_output",
  "call_id": "call_x",
  "output": "{\"agent_id\":\"thread_123\",\"nickname\":\"Hypatia\"}"
}
```

```mermaid
flowchart LR
    A[主 Agent] -- spawn_agent --> B[创建子 Agent]
    B -- 返回 agent_id --> A

    A -- send_input(agent_id) --> C[给子 Agent 派新任务]
    C --> B

    A -- wait_agent(ids) --> D[等待一个或多个子 Agent 到达最终状态]
    B --> D

    A -- close_agent(agent_id) --> E[关闭子 Agent]
    E --> F[Shutdown / Closed]

    A -- resume_agent(agent_id) --> G[恢复已关闭子 Agent]
    G --> B
```

```mermaid
stateDiagram-v2
    [*] --> NotExists
    NotExists --> Running: spawn_agent
    Running --> Running: send_input
    Running --> Final: 自然完成 / 失败
    Running --> Shutdown: close_agent
    Shutdown --> Running: resume_agent
    Final --> Final: wait_agent 只负责观察/等待
    Shutdown --> Shutdown: wait_agent 也可观察到最终状态
```

### spwan_agent

- https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/spec.rs#L1150
    - Only use `spawn_agent` if and only if the user explicitly asks for sub-agents, delegation, or parallel agent work.
        - 为什么skill中的 trigger 不够，因为没有显式的 user input
    - Requests for depth, thoroughness, research, investigation, or detailed codebase analysis do not count as permission to spawn.
        - 为什么读codebase的时候可以；

### wait

- codex-rs/core/src/tools/spec.rs
- wait 指的是 multi-agent 协作里的 wait，不是通用 sleep。它专门等一个或多个子 agent 到 final status。
    - wait_agent 不是靠 sleep 轮询，而是基于状态订阅；

```
{
    "ids": ["<agent_id>", "..."],
    "timeout_ms": 30000
}
```

- 什么时候触发 wait_agent
    - 应该调用 wait_agent 的场景
        - 你下一步必须依赖子 agent 的结果，已经卡住了
        - 你现在就要拿到它的完成状态或最终消息
        - 你在并行派发多个 agent，想看有没有谁先完成，便于继续主流程
    - 不应该急着调 wait_agent 的场景
        - 子 agent 还在后台跑，而你自己还有别的事能做
        - 只是“习惯性想等等看”
        - 想把它当成轮询器不断调用
```
### After you delegate
- Call wait_agent very sparingly. Only call wait_agent when you need the result immediately for the next critical-path step and you are blocked until it returns.
- Do not redo delegated subagent tasks yourself; focus on integrating results or tackling non-overlapping work.
- While the subagent is running in the background, do meaningful non-overlapping work immediately.
- Do not repeatedly wait by reflex.
```
    


```
pub(crate) struct WaitAgentResult {
    pub(crate) status: HashMap<ThreadId, AgentStatus>,
    pub(crate) timed_out: bool,   // 是否超时
}
```

- wait_agent(timeout_ms=...) 的“等待”不是 sleep，而是对 agent 状态流做订阅，并在一个 deadline 内等待 final state。
    - 从 agent 行为上看，会阻塞当前 master agent 的后续推理。它更像：join/await
    - “我下一步必须依赖这个结果”
- 如果 master 不调用 wait_agent，child 完成时也不会失联，因为后台 completion watcher 会自动把 `<subagent_notification>` 注入 parent thread（user role），作为异步完成通知。
    - `<subagent_notification>{"agent_id":"...","status":...}</subagent_notification>`

#### communication of sub agents to master agent

- child 自己先产出 TurnCompleteEvent.last_agent_message。
- 再包装成一个 completion notification（结构化的 json），回流到 parent 的 context 里，而不是直接变成 parent 历史里一条普通 assistant 自然语言消息。（last_agent_message => status.completed 字段）

````
v1 / legacy，大致是 user-role：

  {
    "type": "message",
    "role": "user",
    "content": [
      {
        "type": "input_text",
        "text": "<subagent_notification>{\"agent_path\":\"/root/worker\",\"status\":{\"completed\":\"xxx\"}}</subagent_notification>"
      }
    ]
  }

v2，大致是 assistant-role 的 inter-agent envelope：

  {
    "type": "message",
    "role": "assistant",
    "content": [
      {
        "type": "output_text",
        "text": "{\"author\":\"/root/worker\",\"recipient\":\"/root\",\"other_recipients\":[],\"content\":\"<subagent_notification>{\\\"agent_path\\\":\\\"/root/worker\\\",\\\"status\\\":{\\\"completed\\\":\\
  \"xxx\\\"}}</subagent_notification>\",\"trigger_turn\":false}"
      }
    ]
  }

````

```mermaid
sequenceDiagram
      autonumber
      participant P as Parent Agent
      participant S as spawn_agent Tool
      participant C as Child Thread
      participant M as Child Model Turn
      participant MB as Parent Mailbox
      participant W as wait_agent v2
      participant N as Parent Next Turn

      P->>S: spawn_agent(task_name, message)
      S->>C: create child thread + initial task
      S-->>P: { task_name, nickname? }

      Note over C,M: child runs independently
      C->>M: send child prompt
      M-->>C: final channel message
      C->>C: TurnComplete(last_agent_message)

      C->>C: derive AgentStatus::Completed(last_agent_message)
      C->>MB: InterAgentCommunication {
      C->>MB:   author = child agent_path,
      C->>MB:   recipient = parent agent_path,
      C->>MB:   content = "<subagent_notification>{agent_path,status}</subagent_notification>",
      C->>MB:   trigger_turn = false
      C->>MB: }

      P->>W: wait_agent(timeout_ms)
      W->>MB: subscribe mailbox seq
      MB-->>W: mailbox changed
      W-->>P: { message: "Wait completed.", timed_out: false }

      P->>N: start / continue next parent turn
      N->>MB: drain mailbox
      MB-->>N: assistant-role InterAgentCommunication envelope
      N->>N: record into history
      N->>N: build next model input context
      N-->>P: parent model now sees child completion in context
```

### view_image (Read image) of Codex

- 多模态 agent 能力的来源；（当然首先 base model 是一个vlm，能接受image input）
- 为什么需要一个单独的 tool/function 是 agent 需要自主维护看什么图片，放到 context 中；
    - “prepare image input” 这一步必须由 runtime 做，而不是模型自己做

常见直接调 API，如果你的应用自己已经拿到了图片数据或 URL（“客户端自己已经有图片”），最常见的就是发一条多模态 user message：

```json
{
    "type": "message",
    "role": "user",
    "content": [
      { "type": "input_text", "text": "这张图里有什么？" },
      { "type": "input_image", "image_url": "data:image/png;base64,..." }
    ]
}
```
--------
适用于“图片只存在本地文件系统，需要 agent 先读出来”

```json
{
    "model": "gpt-5.3-codex",
    "instructions": "You are Codex...",
    "input": [
      {
        "type": "message",
        "role": "user",
        "content": [
          { "type": "input_text", "text": "请读取这张图 /abs/path/example.png" }
        ]
      },
      {
          "type": "function_call",
          "name": "view_image",
          "arguments": "{\"path\":\"/abs/path/example.png\"}",
          "call_id": "view-image-call"
      },
      {
        "type": "function_call_output",
        "call_id": "view-image-call",
        "output": [
          {
            "type": "input_image",
            "image_url": "data:image/png;base64,..."
          }
        ]
      }
    ],
    "tools": [
      {
        "type": "function",
        "name": "view_image",
        "description": "View a local image from the filesystem (only use if given a full filepath by the user, and the image isn't already attached to the thread
  context within <image ...> tags).",
        "strict": false,
        "parameters": {
          "type": "object",
          "properties": {
            "path": {
              "type": "string",
              "description": "Local filesystem path to an image file"
            }
          },
          "required": ["path"],
          "additionalProperties": false
        }
      }
    ],
    "tool_choice": "auto",
    "parallel_tool_calls": true,
    "store": false,
    "stream": true,
    "include": []
  }
```

```
"View a local image from the filesystem (only use if given a full filepath by the user, and the image isn't already attached to the thread context within <image ...> tags)." （查看文件系统中的本地图片。仅在用户提供了完整的文件路径，且该图片尚未包含在对话上下文的 <image ...> 标签中时才使用。）
```

view_image 是 Codex 的本地图像桥接工具。它在 tool definition 层向模型暴露“查看本地图片”的能力，在 runtime 层负责完成路径解析、文件读取、缩放/保真策略和 data URL 转换，最终并不是返回普通文本或 JSON，而是把图片作为 function_call_output 中的 input_image 内容项注入后续 responses.create.input。因此，view_image 的本质不是“让模型看图”，而是“把本地文件系统里的图，安全地转成模型真正能看到的图”。
```
{
  "type": "function_call_output",
  "call_id": "call_view_image_x",
  "output": [
    {
      "type": "input_image",
      "image_url": "data:image/png;base64,...",
      "detail": "original"
    }
  ]
}
```

### code mode

```
# 分析这个 codebase 里所有 TODO / FIXME，按文件聚合，顺便判断哪些像技术债，哪些像 bug 提示。

const matches = await tool("rg", {
  pattern: "TODO|FIXME",
  glob: "*.{rs,ts,tsx,js,py}"
});

const grouped = groupByFile(matches);

const results = [];
for (const file of Object.keys(grouped)) {
  const snippet = await tool("read_file", {
    path: file,
    offset: ...,
    limit: ...
  });

  results.push(analyzeTodos(snippet));
}

return summarize(results);

```